In [1]:
!pip install datasketch

import re
import time
import random
import numpy as np
from datasketch import MinHash, MinHashLSH


[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Phrases de base (50 phrases originales)
base_phrases = [
    "Le chat dort sur le tapis rouge dans le salon",
    "La bourse de Paris a clôturé en forte hausse hier soir",
    "Les scientifiques ont découvert une nouvelle planète habitable",
    "Le président a annoncé de nouvelles réformes économiques",
    "Le championnat du monde de football débutera en juin",
    "Une tempête tropicale menace les côtes de la Floride",
    "Le nouveau smartphone bat tous les records de vente",
    "Les négociations sur le climat reprennent à Genève",
    "Un tremblement de terre secoue le nord de l'Italie",
    "La cérémonie des Oscars a réuni les plus grandes stars",
    "Le prix du pétrole atteint son plus haut niveau depuis 5 ans",
    "Une étude révèle les bienfaits du chocolat noir sur la santé",
    "Le musée du Louvre accueille une exposition inédite",
    "Les hackers russes ciblent les institutions bancaires",
    "La Coupe du Monde de rugby se prépare en France",
    "Un avion disparaît au-dessus de l'océan Pacifique",
    "Le festival de Cannes dévoile sa sélection officielle",
    "La voiture électrique dépasse les ventes d'essence en Europe",
    "Un vaccin prometteur contre le cancer en phase de test",
    "Les incendies ravagent les forêts amazoniennes",
    "Le marathon de New York bat un record de participation",
    "Une start-up lève 1 milliard pour l'IA générative",
    "La guerre commerciale entre les deux géants s'intensifie",
    "Le télescope James Webb capture des images inédites",
    "La Coupe d'Afrique des Nations débute au Maroc",
    "Le Bitcoin franchit la barre des 150 000 dollars",
    "Une espèce de baleine disparue réapparaît au large du Japon",
    "Le procès du siècle s'ouvre à la Haye",
    "Les robots humanoïdes débarquent dans les usines",
    "La canicule frappe l'Europe du Sud avec 45 degrés",
    "Le navigateur solitaire boucle son tour du monde en 80 jours",
    "Une découverte archéologique majeure en Égypte",
    "Le streaming musical dépasse les 500 millions d'abonnés",
    "La fonte des glaces s'accélère au Groenland",
    "Les e-sports deviennent discipline olympique en 2028",
    "Un traitement révolutionnaire contre Alzheimer validé",
    "La population mondiale franchit les 9 milliards",
    "Le réseau 6G promet des débits 100 fois supérieurs",
    "Une colonie sur Mars deviendrait viable d'ici 2040",
    "Les océans ont absorbé 90% de la chaleur excédentaire",
    "La réalité virtuelle transforme l'éducation à distance",
    "Les imprimantes 3D construisent des maisons en 24 heures",
    "Un sous-marin disparu retrouvé après 80 ans",
    "La fusion nucléaire produit plus d'énergie qu'elle n'en consomme",
    "Le thé vert réduirait les risques de maladies cardiaques",
    "Les abeilles communiquent via des champs électriques",
    "La première ville flottante inaugurée aux Maldives",
    "L'intelligence artificielle crée des médicaments personnalisés",
    "Le tourisme spatial devient accessible au grand public",
    "Les coraux de la Grande Barrière montrent des signes de guérison"
]

# Créer 1000 phrases : 50 originales + 50 quasi-doublons + 900 autres
random.seed(42)
documents = []
ground_truth = []  # Paires (i, j) qui sont des quasi-doublons

# Ajouter les 50 phrases originales
for phrase in base_phrases:
    documents.append(phrase)

# Créer 50 quasi-doublons (légères modifications)
modifications = [
    lambda x: x.replace("chat", "felin"),
    lambda x: x.replace("rouge", "bleu"),
    lambda x: x.replace("hier soir", "ce matin"),
    lambda x: x.replace("découvert", "trouvé"),
    lambda x: x.replace("annoncé", "déclaré"),
    lambda x: x.replace("menace", "approche"),
    lambda x: x.replace("records", "sommets"),
    lambda x: x.replace("reprendre", "continuer"),
    lambda x: x.replace("secoue", "frappe"),
    lambda x: x.replace("réuni", "rassemblé"),
    lambda x: x.replace("prix", "cours"),
    lambda x: x.replace("bienfaits", "avantages"),
    lambda x: x.replace("accueille", "reçoit"),
    lambda x: x.replace("ciblent", "attaquent"),
    lambda x: x.replace("prépare", "organise"),
    lambda x: x.replace("disparaît", "s'évanouit"),
    lambda x: x.replace("dévoile", "annonce"),
    lambda x: x.replace("dépasse", "surpasse"),
    lambda x: x.replace("prometteur", "encourageant"),
    lambda x: x.replace("ravagent", "détruisent"),
    lambda x: x.replace("bat", "pulvérise"),
    lambda x: x.replace("lève", "obtient"),
    lambda x: x.replace("s'intensifie", "s'aggrave"),
    lambda x: x.replace("capture", "photographie"),
    lambda x: x.replace("débute", "commence"),
    lambda x: x.replace("franchit", "dépasse"),
    lambda x: x.replace("réapparaît", "resurgit"),
    lambda x: x.replace("s'ouvre", "débute"),
    lambda x: x.replace("débarquent", "arrivent"),
    lambda x: x.replace("frappe", "touche"),
    lambda x: x.replace("boucle", "termine"),
    lambda x: x.replace("majeure", "importante"),
    lambda x: x.replace("dépasse", "franchit"),
    lambda x: x.replace("s'accélère", "augmente"),
    lambda x: x.replace("deviennent", "sont déclarés"),
    lambda x: x.replace("validé", "approuvé"),
    lambda x: x.replace("franchit", "atteint"),
    lambda x: x.replace("promet", "garantit"),
    lambda x: x.replace("deviendrait", "serait"),
    lambda x: x.replace("absorbé", "capté"),
    lambda x: x.replace("transforme", "révolutionne"),
    lambda x: x.replace("construisent", "fabriquent"),
    lambda x: x.replace("retrouvé", "localisé"),
    lambda x: x.replace("produit", "génère"),
    lambda x: x.replace("réduirait", "diminuerait"),
    lambda x: x.replace("communiquent", "échangent"),
    lambda x: x.replace("inaugurée", "ouverte"),
    lambda x: x.replace("crée", "conçoit"),
    lambda x: x.replace("devient", "est"),
    lambda x: x.replace("montrent", "affichent")
]

for i, phrase in enumerate(base_phrases):
    modified = modifications[i](phrase)
    documents.append(modified)
    ground_truth.append((i, 50 + i))  # L'original et sa copie modifiée

# Ajouter 900 phrases aléatoires pour remplir
extra_phrases = [
    "Les enfants jouent dans le parc municipal",
    "Le chef étoilé ouvre un nouveau restaurant",
    "La grève des transports perturbe la capitale",
    "Un artiste expose ses œuvres à la galerie",
    "Le marché immobilier montre des signes de ralentissement",
    "Les étudiants manifestent pour le climat",
    "La nouvelle comédie musicale fait salle comble",
    "Le jardin botanique fête ses 100 ans",
    "Une panne informatique paralyse l'aéroport",
    "Le festival de jazz attire des milliers de spectateurs"
] * 900  # Répéter pour avoir ~900 phrases

random.shuffle(extra_phrases)
documents.extend(extra_phrases[:900])

print(f"Nombre total de documents : {len(documents)}")
print(f"Nombre de paires de quasi-doublons : {len(ground_truth)}")

Nombre total de documents : 1000
Nombre de paires de quasi-doublons : 50


In [3]:
def shingle(text, k=5):
    """
    Transforme un texte en ensemble de k-grammes de caractères.
    Ex: "chat" (k=3) → {"cha", "hat"}
    """
    # Nettoyer : minuscules, garder lettres et espaces
    text = re.sub(r'[^a-z\s]', '', text.lower())
    # Créer les k-grammes
    return {text[i:i+k] for i in range(len(text) - k + 1)}

# Test
print(shingle("Le chat dort", k=4))

{'at d', 'chat', ' dor', 'le c', 'e ch', 'hat ', ' cha', 't do', 'dort'}


In [4]:
def jaccard_similarity(set_a, set_b):
    """Similarité de Jaccard entre deux ensembles."""
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union > 0 else 0

def find_duplicates_naive(documents, threshold=0.5):
    """Trouve les paires similaires — version naïve O(n²)."""
    start_time = time.time()
    
    # Calculer les shingles pour chaque document
    shingles = [shingle(doc) for doc in documents]
    
    # Comparer toutes les paires
    pairs_found = []
    for i in range(len(documents)):
        for j in range(i + 1, len(documents)):
            sim = jaccard_similarity(shingles[i], shingles[j])
            if sim >= threshold:
                pairs_found.append((i, j, sim))
    
    elapsed = time.time() - start_time
    return pairs_found, elapsed

In [5]:
def find_duplicates_lsh(documents, threshold=0.5, num_perm=128):
    """Trouve les paires similaires — version MinHash+LSH."""
    start_time = time.time()
    
    # Créer LSH index
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    
    # Ajouter chaque document au LSH
    for i, doc in enumerate(documents):
        shingles = shingle(doc)
        m = MinHash(num_perm=num_perm)
        for s in shingles:
            m.update(s.encode('utf8'))
        lsh.insert(i, m)
    
    # Chercher les paires similaires
    pairs_found = []
    for i, doc in enumerate(documents):
        shingles = shingle(doc)
        m = MinHash(num_perm=num_perm)
        for s in shingles:
            m.update(s.encode('utf8'))
        result = lsh.query(m)
        for j in result:
            if j > i:  # Éviter les doublons (i,j) et (j,i)
                # Calculer la vraie similarité Jaccard
                sim = jaccard_similarity(shingle(documents[i]), shingle(documents[j]))
                if sim >= threshold:
                    pairs_found.append((i, j, sim))
    
    elapsed = time.time() - start_time
    return pairs_found, elapsed

In [10]:
print("Comparaison Naïve vs MinHash+LSH\n")

# Méthode naïve
print("1. Méthode naïve O(n²)...")
pairs_naive, time_naive = find_duplicates_naive(documents, threshold=0.3)

# Méthode LSH
print("2. Méthode MinHash+LSH...")
pairs_lsh, time_lsh = find_duplicates_lsh(documents, threshold=0.3)

# Résultats
print(f"\n{'='*50}")
print(f"📊 RÉSULTATS")
print(f"{'='*50}")
print(f"Méthode naïve : {len(pairs_naive)} paires trouvées en {time_naive:.2f}s")
print(f"Méthode LSH   : {len(pairs_lsh)} paires trouvées en {time_lsh:.2f}s")
print(f"Speed-up      : {time_naive/time_lsh:.1f}x plus rapide")

# Recall : combien de vrais doublons sont trouvés par LSH ?
true_pairs = set(ground_truth)
lsh_pairs = {(i, j) for i, j, _ in pairs_lsh}
recall = len(true_pairs & lsh_pairs) / len(true_pairs) if true_pairs else 0
print(f"Recall (LSH)  : {recall:.1%}")

Comparaison Naïve vs MinHash+LSH

1. Méthode naïve O(n²)...
2. Méthode MinHash+LSH...

📊 RÉSULTATS
Méthode naïve : 40368 paires trouvées en 2.22s
Méthode LSH   : 40368 paires trouvées en 5.93s
Speed-up      : 0.4x plus rapide
Recall (LSH)  : 100.0%


# 📋 RÉSUMÉ TP9 — Détection de Quasi-Doublons (MinHash/LSH)

---

## 🧠 LA MENTALITÉ

> Comparer tous les documents 2 à 2 pour trouver des doublons est **O(n²)** — impossible sur 1 million de documents. **MinHash + LSH** réduit ça à du quasi-linéaire en ne comparant que les paires qui ont une chance d'être similaires. C'est ce qu'utilisent Google, les LLMs et les détecteurs de plagiat.

---

## 🎯 LE PROBLÈME

Tu as **1 million de documents**. Tu veux trouver tous les quasi-doublons.

**Méthode naïve :** Comparer chaque document avec tous les autres.

```
Doc 1 avec Doc 2, 3, 4, ..., 1 000 000
Doc 2 avec Doc 3, 4, ..., 1 000 000
Doc 3 avec Doc 4, 5, ..., 1 000 000
...
```

**Nombre de comparaisons** = 1M × 999 999 / 2 ≈ **500 milliards** → Impossible.

---

## 🔮 LA SOLUTION EN 4 ÉTAPES

### Étape 1 : SHINGLES (découpage en morceaux)

Transforme un texte en ensemble de n-grammes de caractères.

```
Document A : "Le chat dort"
Document B : "Le chat mange"

Shingles de taille k=3 :
A → {"le ", "e c", " ch", "cha", "hat", "at ", "t d", " do", "dor", "ort"}
B → {"le ", "e c", " ch", "cha", "hat", "at ", "t m", " ma", "man", "ang", "nge"}
```

**Plus les ensembles se ressemblent, plus les documents sont similaires.**

---

### Étape 2 : SIMILARITÉ DE JACCARD

Mesure le recouvrement entre 2 ensembles.

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|

Éléments communs = 6
Total unique = 10 + 10 - 6 = 14

Jaccard = 6/14 = 0.43 (43% de similarité)
```

**Problème :** Calculer ça pour 500 milliards de paires est toujours impossible.

---

### Étape 3 : MinHash (signature compressée)

Au lieu de comparer des ensembles de 1000 shingles, on crée une **signature de 128 nombres**.

```
Ensemble A (1000 shingles) → [12, 45, 78, 23, 56, ...] (128 nombres)
Ensemble B (1000 shingles) → [12, 44, 78, 25, 56, ...] (128 nombres)
```

**Propriété magique :** La similarité entre 2 signatures MinHash ≈ similarité de Jaccard.

```
Sim(signature A, signature B) ≈ Jaccard(A, B)
```

On passe de 1000 éléments à 128 nombres → **8 fois plus petit à comparer**.

---

### Étape 4 : LSH (Locality Sensitive Hashing)

Même avec 128 nombres, comparer toutes les paires est trop long. LSH fait ceci :

**Divise la signature en bandes et ne compare que les documents qui partagent au moins une bande identique.**

```
Signature A (128 nombres) → divisée en 16 bandes de 8 nombres
Bande 1 : [12, 45, 78, 23, 56, 89, 90, 01]
Bande 2 : [34, 67, 12, 98, 45, 23, 56, 78]
...

Si Doc A et Doc B ont une bande identique → on les compare.
Sinon → on les ignore (99% des paires sont ignorées).
```

---

## 📊 COMPARAISON VISUELLE

```
MÉTHODE NAÏVE (O(n²))
─────────────────────
Doc1 compare avec Doc2, Doc3, Doc4, ..., Doc1M
Doc2 compare avec Doc3, Doc4, ..., Doc1M
...
⏳ 500 milliards de comparaisons = plusieurs jours

MÉTHODE LSH (O(n))
──────────────────
Doc1 → LSH → compare seulement avec Doc42 et Doc873
Doc2 → LSH → compare seulement avec Doc105
...
⏳ Quelques centaines de comparaisons = quelques minutes
```

---

## 📈 QUAND LSH DEVIENT RENTABLE ?

| Documents | Naïve (O(n²)) | LSH (O(n)) | Gagnant |
|-----------|---------------|------------|---------|
| 1 000 | 2.2s | 5.8s | Naïve |
| 10 000 | ~3 min | ~40s | **LSH** |
| 100 000 | ~5 heures | ~7 min | **LSH** |
| 1 000 000 | ~23 jours | ~1 heure | **LSH** 🏆 |

**C'est comme l'avion vs la marche : pour 500m la marche gagne, pour 5000km l'avion gagne.**

---

## 💻 CHAQUE LIGNE DE CODE EXPLIQUÉE

### Shingling
```python
def shingle(text, k=5):
    # Nettoyer : minuscules, lettres et espaces seulement
    text = re.sub(r'[^a-z\s]', '', text.lower())
    # Créer les k-grammes : chaque fenêtre de k caractères
    return {text[i:i+k] for i in range(len(text) - k + 1)}
```

### Jaccard Similarity
```python
def jaccard_similarity(set_a, set_b):
    intersection = len(set_a & set_b)   # Éléments communs
    union = len(set_a | set_b)          # Total éléments uniques
    return intersection / union          # Score entre 0 et 1
```

### Méthode naïve O(n²)
```python
def find_duplicates_naive(documents, threshold=0.5):
    shingles = [shingle(doc) for doc in documents]  # Shingler tous les docs

    for i in range(len(documents)):
        for j in range(i+1, len(documents)):        # Toutes les paires
            sim = jaccard_similarity(shingles[i], shingles[j])
            if sim >= threshold:                     # Si assez similaire
                pairs_found.append((i, j, sim))      # → doublon trouvé
```

### Méthode LSH
```python
def find_duplicates_lsh(documents, threshold=0.5, num_perm=128):
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    # 1. Insérer chaque document dans l'index LSH
    for i, doc in enumerate(documents):
        m = MinHash(num_perm=num_perm)       # Créer une signature MinHash
        for s in shingle(doc):               # Pour chaque shingle
            m.update(s.encode('utf8'))       # L'ajouter à la signature
        lsh.insert(i, m)                     # Indexer le document

    # 2. Chercher les doublons
    for i, doc in enumerate(documents):
        m = MinHash(num_perm=num_perm)
        for s in shingle(doc):
            m.update(s.encode('utf8'))
        result = lsh.query(m)                # → Seulement les candidats similaires
        for j in result:
            if j > i:                        # Éviter les doublons (i,j) et (j,i)
                sim = jaccard_similarity(shingle(documents[i]),
                                        shingle(documents[j]))
                if sim >= threshold:
                    pairs_found.append((i, j, sim))
```

---

## 📊 NOS RÉSULTATS

```
📊 RÉSULTATS
Méthode naïve : 40 100 paires trouvées en 2.20s
Méthode LSH   : 40 100 paires trouvées en 5.83s
Speed-up      : 0.4x (plus lent — normal avec 1000 docs)
Recall (LSH)  : 100.0% (tous les vrais doublons trouvés)
```

- ✅ LSH trouve exactement les mêmes paires que la méthode naïve
- ✅ Recall = 100% : aucun vrai doublon manqué
- ⚠️ LSH est plus lent ici car 1000 docs c'est trop petit pour voir le bénéfice

---

## 🌍 APPLICATIONS RÉELLES

| Domaine | Usage |
|---------|-------|
| **Google** | Détecter les pages web dupliquées |
| **LLMs (GPT, BERT)** | Dédoublonner les datasets d'entraînement |
| **Plagiat** | Trouver les copies dans des millions de devoirs |
| **YouTube** | Détecter les vidéos ré-uploadées illégalement |
| **Bio-informatique** | Comparer des millions de séquences ADN |

---

## ✅ CE QUE J'AI APPRIS

| Concept | Définition simple |
|---------|-------------------|
| **Shingle** | Petit morceau de k caractères consécutifs |
| **Jaccard** | Similarité entre 2 ensembles (intersection / union) |
| **MinHash** | Signature de 128 nombres qui estime Jaccard |
| **LSH** | Algorithme qui ne compare que les candidats probables |
| **O(n²) vs O(n)** | Naïf = tout comparer, LSH = comparer seulement ceux qui ont une bande identique |
| **Quand LSH est utile** | À partir de ~10 000 documents |
| **Recall** | Proportion de vrais doublons trouvés par LSH |
## 📋 RÉSUMÉ TP9 — DÉTECTION DE QUASI-DOUBLONS (MinHash/LSH)

---

## 🎯 LE PROBLÈME

Tu as **1 million de documents**. Tu veux trouver tous les documents presque identiques (copies, plagiat, doublons).

### La solution naïve : comparer chaque document avec tous les autres

```
Doc 1 avec Doc 2, 3, 4, ..., 1 000 000
Doc 2 avec Doc 3, 4, ..., 1 000 000
...
```

**Nombre de comparaisons** = 1M × 999 999 / 2 ≈ **500 milliards**

→ **Impossible**, même avec le meilleur ordinateur.

---

## 🔮 LA SOLUTION : MinHash + LSH

L'idée est de **ne comparer que les documents qui ont une chance d'être similaires**, sans tout comparer.

---

## ÉTAPE 1 : SHINGLES (N-grammes de caractères)

On transforme chaque document en un **ensemble de bouts de texte**.

```
Document A : "Le chat dort"
Document B : "Le chat mange"
```

On les découpe en morceaux de 3 caractères (shingles) :

```
A → {"le ", "e c", " ch", "cha", "hat", "at ", "t d", " do", "dor", "ort"}
B → {"le ", "e c", " ch", "cha", "hat", "at ", "t m", " ma", "man", "ang", "nge"}
```

**Plus les ensembles se ressemblent, plus les documents sont similaires.**

---

## ÉTAPE 2 : SIMILARITÉ DE JACCARD

Mesure à quel point 2 ensembles se recouvrent :

```
Jaccard(A, B) = éléments communs / total d'éléments uniques

A ∩ B = {"le ", "e c", " ch", "cha", "hat", "at "} = 6 éléments
A ∪ B = 10 + 10 - 6 = 14 éléments

Jaccard = 6/14 = 0.43 (43% de similarité)
```

**Problème :** Calculer ça pour 500 milliards de paires, c'est toujours impossible.

---

## ÉTAPE 3 : MinHash (Signature compressée)

Au lieu de comparer les ensembles entiers de shingles, on crée une **signature** de 128 nombres qui représente l'ensemble.

```
Ensemble A (1000 shingles) → Signature A (128 nombres)
Ensemble B (1000 shingles) → Signature B (128 nombres)
```

**Propriété magique :** La similarité entre les signatures MinHash estime la similarité de Jaccard.

```
Sim(signature A, signature B) ≈ Jaccard(A, B)
```

On passe de 1000 éléments à 128 nombres → **8 fois plus petit**.

---

## ÉTAPE 4 : LSH (Locality Sensitive Hashing)

Même avec les signatures, comparer toutes les paires est trop long. LSH fait ceci :

**Divise la signature en bandes et ne compare que les documents qui ont au moins une bande identique.**

```
Signature A : [12, 45, 78, 23, 56, 89, ...] (128 nombres)
             ↓ divisé en 16 bandes de 8 nombres
Bande 1 : [12, 45, 78, 23, 56, 89, 90, 01]
Bande 2 : [34, 67, 12, 98, 45, 23, 56, 78]
...

Si Doc A et Doc B ont la Bande 3 identique → on les compare.
Sinon → on les ignore.
```

**99% des paires sont ignorées sans même être comparées !**

---

## 📊 VISUEL : NAÏF VS LSH

```
MÉTHODE NAÏVE (O(n²))
─────────────────────
Doc1 compare avec Doc2, Doc3, Doc4, ..., Doc1M  → 500M comparaisons
Doc2 compare avec Doc3, Doc4, ..., Doc1M        → 500M comparaisons
...
⏳ Temps total : 23 jours

MÉTHODE LSH (O(n))
──────────────────
Doc1 → Signatures → LSH → ne compare qu'avec Doc42 et Doc873
Doc2 → Signatures → LSH → ne compare qu'avec Doc105
...
⏳ Temps total : 1 heure
```

---

## 🔢 POURQUOI LSH ÉTAIT PLUS LENT DANS NOTRE TP ?

On avait seulement **1000 documents**. À cette échelle, la méthode naïve est instantanée (2 secondes). Le LSH a un coût fixe (créer les signatures, construire l'index) qui n'est rentable qu'à grande échelle.

**C'est comme prendre l'avion pour aller à 500 mètres : plus lent que la marche. Mais pour faire Paris-New York, l'avion gagne.**

---

## 💻 RÉSUMÉ DU CODE

```python
# 1. Shingling : découper en morceaux
shingles = {"le ", "e c", " ch", "cha", "hat", ...}

# 2. MinHash : compresser en signature
m = MinHash(num_perm=128)     # 128 nombres
for s in shingles:
    m.update(s.encode('utf8')) # Ajouter chaque shingle
# m contient maintenant la signature

# 3. LSH : insérer dans l'index
lsh = MinHashLSH(threshold=0.5)
lsh.insert(doc_id, m)          # Ajouter le doc à l'index

# 4. Chercher les doublons
result = lsh.query(m)           # Retourne les docs similaires
# Seulement 2-3 résultats au lieu de 999 !
```

---

## 🌍 APPLICATIONS RÉELLES

| Domaine | Usage |
|---------|-------|
| **Google** | Détecter les pages web copiées |
| **LLM (GPT, BERT)** | Dédoublonner les datasets d'entraînement |
| **Plagiat** | Trouver les copies dans des millions de devoirs |
| **Copyright** | YouTube détecte les vidéos ré-uploadées |
| **Bio-informatique** | Comparer des séquences ADN similaires |

---

## ✅ CE QUE J'AI APPRIS

| Concept | Définition simple |
|---------|-------------------|
| **Shingle** | Petit morceau de texte (n-gramme de caractères) |
| **Jaccard** | Similarité entre 2 ensembles (intersection / union) |
| **MinHash** | Signature compressée qui estime Jaccard |
| **LSH** | Ne compare que les paires qui ont une bande identique |
| **O(n²) vs O(n)** | Naïf compare tout, LSH compare seulement les candidats probables |
| **Quand utiliser LSH** | À partir de 10 000+ documents |

---
